# Modelos e Avaliação

Especificação, estimação e avaliação out-of-sample dos modelos competidores.

| Modelo | Especificação |
|--------|---------------|
| M0 — Random Walk | σ²ₜ = RV_{t-1} |
| M1 — GARCH(1,1) | σ²ₜ = ω + α·ε²_{t-1} + β·σ²_{t-1} |
| M2 — GJR-GARCH-X(1,1) | σ²ₜ = ω + α·ε²_{t-1} + γ_GJR·ε²_{t-1}·I(ε<0) + β·σ²_{t-1} + δ·ΔSVI_{t-1} |
| M3 — GJR-GARCH(1,1) | σ²ₜ = ω + α·ε²_{t-1} + γ_GJR·ε²_{t-1}·I(ε<0) + β·σ²_{t-1} |

**Inovações Student-t** em M1, M2 e M3 (M2/M3 usam a biblioteca [`gjr-garch-x`](https://github.com/studiofarzulla/gjr-garch-x), que só suporta t; M1 ajustado a t por consistência distribucional).

O **M3** permite atribuição limpa do efeito do SVI: M3 isola o efeito assimetria (GJR) e M2 acrescenta o termo δ·ΔSVI a esse mesmo modelo. Assim:
- **M1 vs M3** mede o ganho da assimetria.
- **M3 vs M2** mede o ganho puro do SVI condicional à assimetria.

Requer os CSVs gerados por `01_coleta_dados.ipynb` na pasta `./data/`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from arch import arch_model
from arch.bootstrap import MCS
from gjr_garch_x import estimate_gjr_garch_x

CAMINHO = "./data/"

# Bases salvas

In [2]:
df_acoes = pd.read_csv("./data/acoes_elegiveis.csv")
df_trends = pd.read_csv("./data/google_trends_svi.csv", parse_dates=["semana"])
df_precos = pd.read_csv("./data/precos_semanais.csv", parse_dates=["semana"])
df_log = pd.read_csv("./data/log_coleta_svi.csv")
df_adf = pd.read_csv("./data/log_adf_svi.csv")
df_final = pd.read_csv("./data/base_final_tcc.csv", parse_dates=["semana"])

df_previsoes = pd.read_csv("./data/previsoes_consolidadas.csv", parse_dates=["semana"])
df_resumo_params = pd.read_csv("./data/resumo_parametros.csv")

# Especificação dos Modelos Competidores

## MODELO 0: PASSEIO ALEATÓRIO (Random Walk)
σ̂²_t = RV_{t-1}

In [3]:
def passeio_aleatorio(grupo):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv"]]
        .copy()
    )
    df_t["rv_pred_m0"] = df_t["rv"].shift(1)
    return df_t.dropna(subset=["rv_pred_m0"])

## MODELO 1: GARCH(1,1) PADRÃO
σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}

Inovações Student-t (para consistência distribucional com M2 e M3, cuja biblioteca só suporta t).
Janela móvel de tamanho fixo: 52 * 3 semanas.

In [4]:
def garch_padrao(grupo, col_end="ret_semanal", janela=52 * 3):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_end]]
        .copy()
    )
    n, previsoes = len(df_t), []

    for t in range(janela, n):
        treino = df_t[col_end].iloc[t - janela : t].values.astype(float) * 100
        try:
            fit = arch_model(treino, mean="Constant", vol="GARCH",
                             p=1, q=1, dist="t").fit(disp="off", show_warning=False)
            var_pred = fit.forecast(horizon=1, reindex=False).variance.values[-1, 0] / 10_000
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                col_end:         df_t[col_end].iloc[t],
                "rv_pred_m1": var_pred,
                "omega_m1":   fit.params.get("omega",    np.nan),
                "alpha_m1":   fit.params.get("alpha[1]", np.nan),
                "beta_m1":    fit.params.get("beta[1]",  np.nan),
                "nu_m1":      fit.params.get("nu",       np.nan),
            })
        except Exception as e:
            print(f"  [ERRO GARCH t={t}]: {e}")
            previsoes.append({"semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t], col_end: df_t[col_end].iloc[t],
                              "rv_pred_m1": np.nan, "omega_m1": np.nan, "alpha_m1": np.nan,
                              "beta_m1": np.nan, "nu_m1": np.nan})

    return pd.DataFrame(previsoes)

## MODELO 2: GJR-GARCH-X(1,1)
$\sigma^2_t = \omega + \alpha \cdot \varepsilon^2_{t-1} + \gamma_{GJR} \cdot \varepsilon^2_{t-1} \cdot \mathbb{I}(\varepsilon_{t-1}<0) + \beta \cdot \sigma^2_{t-1} + \delta \cdot \Delta SVI_{t-1}$

Janela móvel de tamanho fixo: 52 * 3 semanas. Inovações Student-t.

---
**Mudanças em relação ao Modelo 1:**

1. **ΔSVI agora entra na equação da variância** (δ·ΔSVI_{t-1}), não mais na média. Captura efeito direto da atenção do investidor sobre a volatilidade condicional.
2. **Termo de assimetria GJR** (γ_GJR·ε²·I(ε<0)) captura o efeito alavancagem: choques negativos podem elevar a volatilidade mais que choques positivos do mesmo módulo.

**Implementação** (biblioteca [`gjr-garch-x`](https://github.com/studiofarzulla/gjr-garch-x)):

- A biblioteca trata o exógeno como **contemporâneo** a σ²_t. Para representar ΔSVI_{t-1} dentro da janela de estimação, passamos `svi_diff.shift(1)` — assim o que entra na variância em cada t é o ΔSVI defasado uma semana, evitando look-ahead bias.
- A lib **não possui método `.forecast()`**. A previsão de σ²_t um passo à frente é calculada manualmente pela equação fechada usando ω, α, β, γ_GJR, δ estimados, o último resíduo da janela (ε_{t-1} = `res.residuals.iloc[-1]`), a última variância condicional (σ²_{t-1} = `res.volatility.iloc[-1]**2`) e ΔSVI_{t-1} = `svi_diff.iloc[t-1]`.

**Saídas reportadas:** ω, α, β, ν (graus de liberdade da t), γ_GJR (com p-valor), δ (`gamma_svi`, com p-valor). A persistência sob choque simétrico é α + β + γ_GJR/2.

In [5]:
def gjr_garch_x_svi(grupo, col_end="ret_semanal", col_exog="svi_diff", janela=52 * 3):
    """
    GJR-GARCH-X(1,1) com ΔSVI na equação da variância.

    σ²_t = ω + α·ε²_{t-1} + γ_GJR·ε²_{t-1}·I(ε_{t-1}<0) + β·σ²_{t-1} + δ·ΔSVI_{t-1}

    A biblioteca `gjr-garch-x` trata o exógeno como contemporâneo a σ²_t. Para que
    o que entre na variância no tempo t seja ΔSVI_{t-1} (defasado 1 semana), passamos
    `svi_diff.shift(1)` como exógeno. Inovações Student-t (única opção da lib).
    """
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_end, col_exog]]
        .dropna(subset=[col_exog])
        .reset_index(drop=True)
        .copy()
    )
    df_t["svi_lag1"] = df_t[col_exog].shift(1)
    n, previsoes = len(df_t), []

    if n <= janela:
        print(f"  [AVISO] Obs. insuficientes ({n}), retornando vazio.")
        return pd.DataFrame()

    for t in range(janela, n):
        ret_series = pd.Series(
            df_t[col_end].iloc[t - janela : t].values.astype(float) * 100,
            index=df_t["semana"].iloc[t - janela : t],
        )
        exog_df = pd.DataFrame(
            {"svi_diff": df_t["svi_lag1"].iloc[t - janela : t].values.astype(float)},
            index=ret_series.index,
        )

        try:
            res = estimate_gjr_garch_x(ret_series, exog_df, method="SLSQP", verbose=False)

            if not res.converged:
                raise RuntimeError("nao convergiu")

            omega    = res.params["omega"]
            alpha    = res.params["alpha"]
            beta     = res.params["beta"]
            gamma_gjr = res.params["gamma"]
            delta    = res.exog_effects.get("svi_diff", np.nan)
            gjr_pval = res.pvalues.get("gamma",    np.nan)
            svi_pval = res.pvalues.get("svi_diff", np.nan)

            eps_tm1   = res.residuals.iloc[-1]
            sig2_tm1  = res.volatility.iloc[-1] ** 2
            dsvi_tm1  = df_t[col_exog].iloc[t - 1]

            sig2_t = (
                omega
                + alpha * eps_tm1 ** 2
                + gamma_gjr * (eps_tm1 ** 2) * (eps_tm1 < 0)
                + beta * sig2_tm1
                + delta * dsvi_tm1
            )
            # Floor consistente com o recursion interno da lib (linha 339 do source):
            # variância não pode ficar <=0 (acontece quando δ·ΔSVI < 0 supera os demais termos).
            sig2_t = max(sig2_t, 1e-8)
            var_pred = sig2_t / 10_000

            previsoes.append({
                "semana":             df_t["semana"].iloc[t],
                "rv":                 df_t["rv"].iloc[t],
                col_end:              df_t[col_end].iloc[t],
                "rv_pred_m2":         var_pred,
                "omega_m2":           omega,
                "alpha_m2":           alpha,
                "beta_m2":            beta,
                "nu_m2":              res.params.get("nu", np.nan),
                "gamma_gjr_m2":       gamma_gjr,
                "gamma_gjr_pval_m2":  gjr_pval,
                "gamma_svi":          delta,
                "gamma_svi_pval":     svi_pval,
                "status_m2":          "ok",
            })
        except Exception as e:
            previsoes.append({
                "semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t],
                col_end: df_t[col_end].iloc[t],
                "rv_pred_m2": np.nan, "omega_m2": np.nan, "alpha_m2": np.nan,
                "beta_m2": np.nan, "nu_m2": np.nan,
                "gamma_gjr_m2": np.nan, "gamma_gjr_pval_m2": np.nan,
                "gamma_svi": np.nan, "gamma_svi_pval": np.nan,
                "status_m2": str(e),
            })

    return pd.DataFrame(previsoes)

## MODELO 3: GJR-GARCH(1,1) (sem SVI)
$\sigma^2_t = \omega + \alpha \cdot \varepsilon^2_{t-1} + \gamma_{GJR} \cdot \varepsilon^2_{t-1} \cdot \mathbb{I}(\varepsilon_{t-1}<0) + \beta \cdot \sigma^2_{t-1}$

Mesma biblioteca e otimizador do M2, mas sem o termo exógeno (`exog_vars=None`). Inovações Student-t.

---
**Motivação metodológica — atribuição limpa do efeito do SVI:**

- **M1 vs M3**: isola o efeito da **assimetria** (GJR) sobre a volatilidade, sem confundir com SVI.
- **M3 vs M2**: isola o efeito **puro do SVI** condicional à assimetria já modelada.
- **M1 vs M2**: confunde os dois efeitos — útil apenas para comparação global.

Sem o M3, qualquer ganho do M2 sobre o M1 poderia ser creditado tanto ao SVI quanto ao GJR.

In [6]:
def gjr_garch_puro(grupo, col_end="ret_semanal", janela=52 * 3):
    """
    GJR-GARCH(1,1) puro, sem variável exógena na variância.

    σ²_t = ω + α·ε²_{t-1} + γ_GJR·ε²_{t-1}·I(ε_{t-1}<0) + β·σ²_{t-1}

    Estimado com `gjr-garch-x` passando `exog_vars=None`, mesma rotina/otimizador do M2.
    Forecast manual 1-passo (a lib não tem `.forecast()`). Inovações Student-t.
    """
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_end]]
        .copy()
    )
    n, previsoes = len(df_t), []

    if n <= janela:
        return pd.DataFrame()

    for t in range(janela, n):
        ret_series = pd.Series(
            df_t[col_end].iloc[t - janela : t].values.astype(float) * 100,
            index=df_t["semana"].iloc[t - janela : t],
        )
        try:
            res = estimate_gjr_garch_x(ret_series, None, method="SLSQP", verbose=False)

            if not res.converged:
                raise RuntimeError("nao convergiu")

            omega     = res.params["omega"]
            alpha     = res.params["alpha"]
            beta      = res.params["beta"]
            gamma_gjr = res.params["gamma"]
            gjr_pval  = res.pvalues.get("gamma", np.nan)

            eps_tm1  = res.residuals.iloc[-1]
            sig2_tm1 = res.volatility.iloc[-1] ** 2

            sig2_t = (
                omega
                + alpha * eps_tm1 ** 2
                + gamma_gjr * (eps_tm1 ** 2) * (eps_tm1 < 0)
                + beta * sig2_tm1
            )
            # Floor consistente com o recursion interno da lib (lib aplica 1e-8 a cada t).
            sig2_t = max(sig2_t, 1e-8)
            var_pred = sig2_t / 10_000

            previsoes.append({
                "semana":            df_t["semana"].iloc[t],
                "rv":                df_t["rv"].iloc[t],
                col_end:             df_t[col_end].iloc[t],
                "rv_pred_m3":        var_pred,
                "omega_m3":          omega,
                "alpha_m3":          alpha,
                "beta_m3":           beta,
                "nu_m3":             res.params.get("nu", np.nan),
                "gamma_gjr_m3":      gamma_gjr,
                "gamma_gjr_pval_m3": gjr_pval,
                "status_m3":         "ok",
            })
        except Exception as e:
            previsoes.append({
                "semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t],
                col_end: df_t[col_end].iloc[t],
                "rv_pred_m3": np.nan, "omega_m3": np.nan, "alpha_m3": np.nan,
                "beta_m3": np.nan, "nu_m3": np.nan,
                "gamma_gjr_m3": np.nan, "gamma_gjr_pval_m3": np.nan,
                "status_m3": str(e),
            })

    return pd.DataFrame(previsoes)

# Avaliação de Previsão e Inferência (Modelos Aninhados)
```
Janela de treino:   svi_diff[ t-156 … t-1 ]  →  estima γ
Previsão:           svi_diff[ t-1 ]           →  usa γ para prever σ²_t



In [7]:
# ── Proporção de zeros no SVI por ticker ─────────────────────────────
# Calculada aqui para uso nos três cenários de avaliação.
# O filtro NÃO é aplicado sobre df_final — todos os tickers entram
# na modelagem; o corte é feito na etapa de avaliação (texto principal
# e apêndice), permitindo comparação direta entre cenários.
# IMPORTANTE: re-execute a célula do Loop Principal após esta alteração
# para regenerar df_previsoes com todos os tickers.
zeros_pct = df_trends.groupby("ticker_b3")["svi"].apply(lambda x: (x == 0).mean())

In [8]:
# ── Configurações ─────────────────────────────────────────────────
JANELA = 52 * 3

NOMES = {
    "qlike_m0": "M0 (RW)",
    "qlike_m1": "M1 (GARCH)",
    "qlike_m2": "M2 (GJR-X)",
    "qlike_m3": "M3 (GJR)",
}

In [9]:
# ── Loop principal ─────────────────────────────────────────────────
resultados, resumo_params = [], []

for ticker, grupo in df_final.groupby("ticker_b3"):
    print(f"► {ticker}", end=" ... ")

    df_t = grupo.sort_values("semana").reset_index(drop=True)

    if len(df_t) <= JANELA:
      print("insuficiente, pulando.")
      continue

    semana_corte = df_t["semana"].iloc[JANELA]

    df_m0 = passeio_aleatorio(grupo)
    df_m1 = garch_padrao(grupo, janela=JANELA)
    df_m2 = gjr_garch_x_svi(grupo, janela=JANELA)
    df_m3 = gjr_garch_puro(grupo, janela=JANELA)

    if df_m2.empty or df_m3.empty:
        print("M2/M3 vazio, pulando.")
        continue

    # ── Merge ──────────────────────────────────────────────────────
    # rv_pred_* = previsão de variância condicional (saída do GARCH)
    # rv        = volatilidade realizada (benchmark para QLIKE)
    cols_m2 = ["semana", "rv_pred_m2", "omega_m2", "alpha_m2", "beta_m2", "nu_m2",
               "gamma_gjr_m2", "gamma_gjr_pval_m2", "gamma_svi", "gamma_svi_pval",
               "status_m2"]
    cols_m3 = ["semana", "rv_pred_m3", "omega_m3", "alpha_m3", "beta_m3", "nu_m3",
               "gamma_gjr_m3", "gamma_gjr_pval_m3", "status_m3"]

    df_tick = (
        df_m1
        .merge(df_m0[["semana", "rv_pred_m0"]],     on="semana", how="left")
        .merge(df_m2[cols_m2],                       on="semana", how="left")
        .merge(df_m3[cols_m3],                       on="semana", how="left")
    )
    df_tick["ticker_b3"] = ticker
    df_tick["split"]     = np.where(
        df_tick["semana"] >= semana_corte, "out-of-sample", "in-sample"
    )
    resultados.append(df_tick)

    # ── Resumo parâmetros ──────────────────────────────────────────
    resumo_params.append({
        "ticker_b3":               ticker,
        "n_previsoes":             len(df_m2),
        # SVI (somente M2)
        "gamma_svi_medio":         df_m2["gamma_svi"].mean(),
        "gamma_svi_pval_medio":    df_m2["gamma_svi_pval"].mean(),
        "pct_signif_5pct_svi":     (df_m2["gamma_svi_pval"] < 0.05).mean() * 100,
        # GJR no M2 e no M3
        "gamma_gjr_m2_medio":      df_m2["gamma_gjr_m2"].mean(),
        "gamma_gjr_m2_pval_medio": df_m2["gamma_gjr_pval_m2"].mean(),
        "pct_signif_5pct_gjr_m2":  (df_m2["gamma_gjr_pval_m2"] < 0.05).mean() * 100,
        "gamma_gjr_m3_medio":      df_m3["gamma_gjr_m3"].mean(),
        "gamma_gjr_m3_pval_medio": df_m3["gamma_gjr_pval_m3"].mean(),
        "pct_signif_5pct_gjr_m3":  (df_m3["gamma_gjr_pval_m3"] < 0.05).mean() * 100,
        # Persistência (sob choque simétrico: α + β + γ_GJR/2)
        "persistencia_m1":         (df_m1["alpha_m1"] + df_m1["beta_m1"]).mean(),
        "persistencia_m2":         (df_m2["alpha_m2"] + df_m2["beta_m2"]
                                    + df_m2["gamma_gjr_m2"] / 2).mean(),
        "persistencia_m3":         (df_m3["alpha_m3"] + df_m3["beta_m3"]
                                    + df_m3["gamma_gjr_m3"] / 2).mean(),
    })
    print("✓")

# ── Consolida ──────────────────────────────────────────────────────
df_previsoes     = pd.concat(resultados, ignore_index=True)
df_resumo_params = pd.DataFrame(resumo_params).set_index("ticker_b3")

# ── Salva ──────────────────────────────────────────────────────────
df_previsoes.to_csv(CAMINHO + "previsoes_consolidadas.csv", index=False)
df_resumo_params.to_csv(CAMINHO + "resumo_parametros.csv")

► AALR3 ... ✓
► ABCB4 ... ✓
► ABEV3 ... ✓
► AERI3 ... ✓
► AFLT3 ... ✓
► AGRO3 ... ✓
► AHEB3 ... ✓
► ALOS3 ... ✓
► ALPA3 ... ✓
► ALPA4 ... ✓
► ALPK3 ... ✓
► ALUP11 ... ✓
► ALUP3 ... ✓
► ALUP4 ... ✓
► AMAR3 ... ✓
► AMBP3 ... ✓
► AMER3 ... ✓
► ANIM3 ... ✓
► ASAI3 ... ✓
► AVLL3 ... ✓
► AXIA3 ... ✓
► AXIA5 ... ✓
► AXIA6 ... ✓
► AZEV3 ... ✓
► AZEV4 ... ✓
► AZZA3 ... ✓
► B3SA3 ... ✓
► BALM4 ... ✓
► BAZA3 ... ✓
► BBAS3 ... ✓
► BBDC3 ... ✓
► BBDC4 ... ✓
► BBSE3 ... ✓
► BDLL4 ... ✓
► BEEF3 ... ✓
► BEES3 ... ✓
► BEES4 ... ✓
► BGIP3 ... ✓
► BGIP4 ... ✓
► BHIA3 ... ✓
► BIOM3 ... ✓
► BMEB3 ... ✓
► BMEB4 ... ✓
► BMGB4 ... ✓
► BMIN4 ... ✓
► BMKS3 ... ✓
► BMOB3 ... ✓
► BNBR3 ... ✓
► BOBR4 ... ✓
► BPAC11 ... ✓
► BPAC3 ... ✓
► BPAC5 ... ✓
► BRAP3 ... ✓
► BRAP4 ... ✓
► BRAV3 ... ✓
► BRKM3 ... ✓
► BRKM5 ... ✓
► BRKM6 ... ✓
► BRSR3 ... ✓
► BRSR5 ... ✓
► BRSR6 ... ✓
► BSLI3 ... ✓
► CAMB3 ... ✓
► CAML3 ... ✓
► CASH3 ... ✓
► CBEE3 ... ✓
► CEAB3 ... ✓
► CEBR3 ... ✓
► CEBR5 ... ✓
► CEBR6 ... ✓
► CEDO3 ... ✓
► CE

In [10]:
# Preserva versão completa (usada na comparação diagnóstica)
mask_erro_m2 = df_previsoes["status_m2"].fillna("").ne("ok") & df_previsoes["status_m2"].notna()
mask_erro_m3 = df_previsoes["status_m3"].fillna("").ne("ok") & df_previsoes["status_m3"].notna()
ticker_erro_m2 = df_previsoes.loc[mask_erro_m2, "ticker_b3"].unique()
ticker_erro_m3 = df_previsoes.loc[mask_erro_m3, "ticker_b3"].unique()
ticker_com_erro = set(ticker_erro_m2) | set(ticker_erro_m3)

df_previsoes_raw = df_previsoes.copy()  # inclui tickers com erro
df_previsoes     = df_previsoes[~df_previsoes.ticker_b3.isin(ticker_com_erro)]  # sem erros

print(f"Tickers em df_previsoes_raw   : {df_previsoes_raw['ticker_b3'].nunique()}")
print(f"Tickers com erro M2           : {len(ticker_erro_m2)}")
print(f"Tickers com erro M3           : {len(ticker_erro_m3)}")
print(f"Tickers com erro (M2 ∪ M3)    : {len(ticker_com_erro)}")
print(f"Tickers em df_previsoes       : {df_previsoes['ticker_b3'].nunique()}")

Tickers em df_previsoes_raw   : 323
Tickers com erro M2           : 58
Tickers com erro M3           : 16
Tickers com erro (M2 ∪ M3)    : 66
Tickers em df_previsoes       : 257


# Diagnóstico: Comparação de Estratégias de Filtro

São testadas quatro estratégias e comparados: número de tickers rodados,
tickers com erro (singular matrix), tickers na comparação final, QLIKE e MCS.

| # | Estratégia | Filtro zeros | Remove singular matrix |
|---|------------|-------------|------------------------|
| 1 | Singular matrix removido (sem filtro zeros) | nenhum | sim |
| 2 | Filtro ≥70% + remove singular matrix | ≥70% | sim |
| 3 | Filtro ≥50% + remove singular matrix | ≥50% | sim |
| 4 | Sem filtro (NaN via `.dropna()`) | nenhum | não |

In [ ]:
# ── Função de perda QLIKE ───────────────────────────────────────
def qlike(rv, pred):
    mask = (rv > 0) & (pred > 0)
    r    = np.where(mask, rv / pred, np.nan)
    return np.where(mask, r - np.log(r) - 1, np.nan)


# ── Função de comparação por cenário ────────────────────────────
def comparar_cenario(df_prev, threshold_zeros=None, remove_singular=True,
                     label="", df_params=None, txt_path=None):
    """
    threshold_zeros : float | None
        None  -> sem filtro de zeros
        0.70  -> remove tickers com >= 70% de zeros no SVI
        0.50  -> remove tickers com >= 50% de zeros no SVI
    remove_singular : bool
        True  -> remove explicitamente tickers com erro em M2 ou M3
        False -> mantém; NaN descartados pelo .dropna() no MCS
    df_params : DataFrame resumo_parametros (opcional)
        Exibe Top 5 por % de janelas com gamma_svi e gamma_gjr_m2 significativos.
    txt_path : str | None
        Se fornecido, salva (append) o resultado neste arquivo .txt.
    """
    df_sub = df_prev.copy()
    n_inicial = df_sub["ticker_b3"].nunique()

    erro_m2 = df_sub["rv_pred_m2"].isna()
    erro_m3 = df_sub["rv_pred_m3"].isna()
    tickers_com_erro = df_sub.loc[erro_m2 | erro_m3, "ticker_b3"].unique()
    n_erro = len(tickers_com_erro)

    if threshold_zeros is not None:
        tickers_zeros_ok = zeros_pct[zeros_pct < threshold_zeros].index
        df_sub = df_sub[df_sub["ticker_b3"].isin(tickers_zeros_ok)]
    n_apos_zeros = df_sub["ticker_b3"].nunique()
    n_erro_rest  = df_sub.loc[df_sub["rv_pred_m2"].isna() | df_sub["rv_pred_m3"].isna(),
                              "ticker_b3"].nunique()

    if remove_singular:
        err_sub = df_sub.loc[df_sub["rv_pred_m2"].isna() | df_sub["rv_pred_m3"].isna(),
                             "ticker_b3"].unique()
        df_sub  = df_sub[~df_sub["ticker_b3"].isin(err_sub)]

    n_comparacao = df_sub["ticker_b3"].nunique()

    df_oos_sub = df_sub.query("split == 'out-of-sample'").copy()
    df_oos_sub["qlike_m0"] = qlike(df_oos_sub["rv"], df_oos_sub["rv_pred_m0"])
    df_oos_sub["qlike_m1"] = qlike(df_oos_sub["rv"], df_oos_sub["rv_pred_m1"])
    df_oos_sub["qlike_m2"] = qlike(df_oos_sub["rv"], df_oos_sub["rv_pred_m2"])
    df_oos_sub["qlike_m3"] = qlike(df_oos_sub["rv"], df_oos_sub["rv_pred_m3"])

    oos_losses = (
        df_oos_sub[list(NOMES)]
        .dropna()
        .rename(columns=NOMES)
        .reset_index(drop=True)
    )
    n_obs           = len(oos_losses)
    n_obs_descard   = len(df_oos_sub) - n_obs

    mcs_res = MCS(oos_losses, size=0.10, block_size=5, reps=1000, seed=42)
    mcs_res.compute()
    df_mcs_res = (
        mcs_res.pvalues
        .rename(columns={"Pvalue": "p_valor"})
        .assign(no_conjunto=lambda d: d["p_valor"] >= 0.10)
        .sort_values("p_valor", ascending=False)
    )

    # ── Monta bloco de texto ─────────────────────────────────────
    sep = "═" * 60
    L = []
    L += [f"\n{sep}", f"  {label}", sep]
    L += [
        f"  Tickers em df_previsoes              : {n_inicial}",
        f"  Com erro (M2 ou M3)                  : {n_erro}",
    ]
    if threshold_zeros is not None:
        L.append(f"  Após filtro zeros ≥{threshold_zeros:.0%}              : {n_apos_zeros}  ({n_erro_rest} com erro)")
    L += [
        f"  Na comparação final                   : {n_comparacao}",
        f"  Obs. OOS usadas / descartadas        : {n_obs} / {n_obs_descard}",
        "",
        "  QLIKE médio:",
    ]
    for modelo, val in oos_losses.mean().items():
        L.append(f"    {modelo:<15}: {val:.6f}")
    L += ["", "  MCS (α=10%):", df_mcs_res.to_string(),
          f"\n  Conjunto de confiança : {list(mcs_res.included)}",
          f"  Excluídos             : {list(mcs_res.excluded)}"]

    # ── Top 5 gamma significativo ────────────────────────────────
    if df_params is not None:
        p = df_params.copy()
        if "ticker_b3" in p.columns:
            p = p.set_index("ticker_b3")
        p = p[p.index.isin(df_sub["ticker_b3"].unique())]

        n_sig_svi = (p["pct_signif_5pct_svi"] >= 50).sum()
        n_sig_gjr_m2 = (p["pct_signif_5pct_gjr_m2"] >= 50).sum()
        L += [
            "",
            f"  γ_SVI (δ) significativo em ≥50% das janelas:",
            f"    {n_sig_svi}/{len(p)} tickers",
            "",
            "  Top 5 por % de janelas significativas (γ_SVI):",
            p["pct_signif_5pct_svi"].sort_values(ascending=False).head().to_string(),
            "",
            f"  γ_GJR (M2) significativo em ≥50% das janelas:",
            f"    {n_sig_gjr_m2}/{len(p)} tickers",
            "",
            "  Top 5 por % de janelas significativas (γ_GJR no M2):",
            p["pct_signif_5pct_gjr_m2"].sort_values(ascending=False).head().to_string(),
        ]

    resultado = "\n".join(str(x) for x in L)
    print(resultado)

    if txt_path is not None:
        with open(txt_path, "a", encoding="utf-8") as fh:
            fh.write(resultado + "\n")

    return df_mcs_res, oos_losses.mean()

In [ ]:
TXT = CAMINHO + "diagnostico_filtros.txt"
open(TXT, "w", encoding="utf-8").close()  # limpa arquivo

df_mcs_1, qlike_1 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=None, remove_singular=True,
    label="1. Singular matrix removido (sem filtro zeros)",
    df_params=df_resumo_params, txt_path=TXT,
)

df_mcs_2, qlike_2 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=0.70, remove_singular=True,
    label="2. Filtro ≥70% + singular matrix removido",
    df_params=df_resumo_params, txt_path=TXT,
)

df_mcs_3, qlike_3 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=0.50, remove_singular=True,
    label="3. Filtro ≥50% + singular matrix removido",
    df_params=df_resumo_params, txt_path=TXT,
)

df_mcs_4, qlike_4 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=None, remove_singular=False,
    label="4. Sem filtro (NaN via .dropna())",
    df_params=df_resumo_params, txt_path=TXT,
)

print(f"\n✅ Resultados salvos em {TXT}")



════════════════════════════════════════════════════════════
  1. Singular matrix removido (sem filtro zeros)
════════════════════════════════════════════════════════════
  Tickers em df_previsoes              : 323
  Com erro (singular matrix)           : 29
  Na comparação final                   : 294
  Obs. OOS usadas / descartadas        : 29070 / 1212

  QLIKE médio:
    M0 (RW)        : 38.981838
    M1 (GARCH)     : 0.794214
    M2 (GARCH-X)   : 0.798090

  MCS (α=10%):
              p_valor  no_conjunto
Model name                        
M1 (GARCH)      1.000         True
M2 (GARCH-X)    0.342         True
M0 (RW)         0.008        False

  Conjunto de confiança : ['M1 (GARCH)', 'M2 (GARCH-X)']
  Excluídos             : ['M0 (RW)']

  γ (ΔSVI) significativo em ≥50% das janelas:
    62/294 tickers

  Top 5 por % de janelas significativas:
ticker_b3
BGIP4    100.0
ESTR4    100.0
MWET4    100.0
BMGB4    100.0
OSXB3    100.0

═══════════════════════════════════════════════════